In [1]:
#| default_exp rest

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
#| export
from rest.core import init_instance, generate
singleton, model_path = init_instance()

In [5]:
model_path = 'pelevin'

In [6]:
#| export
seq_length = 1024

model_path = f'./models/large/{model_path}'
import torch
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path, pad_token_id = 50256)
model = GPT2LMHeadModel.from_pretrained(model_path,torch_dtype=torch.bfloat16)
model.config.pad_token_id = model.config.eos_token_id
model.cuda()
model.eval();

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [7]:
sum(p.numel() for p in model.parameters())

774030080

In [8]:
#| export
import threading
lock = threading.RLock()

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    with lock:
        return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [9]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

/usr/local/lib/python3.8/dist-packages/transformers/generation/utils.py:1255: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation)
  warnings.warn(


CPU times: user 2 s, sys: 110 ms, total: 2.11 s
Wall time: 2.08 s


[' — чистый Воланд! Покуда от человека зависит, т. е. он может ошибаться и ошибаться, пока он ходит по городу и дышит воздухом, он хоть Лев, хоть не Лев.',
 ' – просто Кеша, затраханный баблосом игрок.',
 ' ты Павлик Морозов. Какой Павка Морозов?» Я засмеялась. И тогда он снова обернулся. Оглядел меня со всех сторон. Спросил: «Ты знаешь, кто такой Павкин?» – «Знаю, Лев Николаевич.',
 ' завхоз фабрики “ Палладиум”!» – быстро прикинул Гоша. Потом – музыка смолкла, в дверь постучали. Гошка встал и пошел открывать. На пороге стоял плотный мужик в пижаме и тапочках.']